# DebateFloor — GRPO Training Notebook (Unsloth + QLoRA)
### Meta PyTorch × Scaler Hackathon Grand Finale | April 25–26, 2026

| | |
|---|---|
| **Model** | `Qwen/Qwen2.5-0.5B-Instruct` via **Unsloth** 4-bit QLoRA (MR-3) |
| **Runtime** | **T4 GPU** (free Colab tier), ~15 min for 100 episodes |
| **Method** | TRL `GRPOTrainer` with live HTTP environment reward (MR-2) |
| **Reward** | From `POST /step` on the local DebateFloor env server |
| **Output** | `docs/reward_curve.svg`, `docs/component_shift.svg`, `reports/training_summary.json`, checkpoint pushed to HF Hub |

---
**Run all cells top to bottom. Restart runtime after Cell 1, then continue from Cell 2.**

## Cell 1 — Install & clone
Run once. **Restart runtime after this cell.**

In [ ]:
import os

if not os.path.isdir("/content/debateFloor"):
    !git clone https://github.com/AniketAslaliya/debateFloor.git

%cd /content/debateFloor

# Core: TRL + model stack
!pip install -q trl>=0.12.0 transformers>=4.46.0 peft accelerate datasets wandb requests matplotlib bitsandbytes

# Unsloth (MR-3) — 4-bit QLoRA
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# App server: FastAPI + OpenEnv (REQUIRED — app/ imports openenv.core.*)
!pip install -q fastapi uvicorn pydantic openenv-core

print("\nDone. Runtime -> Restart session, then run the next cell (re-clone + start server).")

## Cell 2 — Configure & start environment server
**Edit values below**, then run. Starts the DebateFloor env server locally for fast reward (~5ms vs ~500ms to remote Space).

In [ ]:
import os, sys, subprocess, time, requests, importlib

REPO_DIR = '/content/debateFloor'
PIPS = [sys.executable, '-m', 'pip', 'install', '-q']


def ensure_repo_and_deps():
    """Clone if missing. Install deps only when needed (openenv-core is the usual Colab gotcha)."""
    fresh = False
    if not os.path.isdir(REPO_DIR):
        print('Cloning debateFloor (repo not found after restart or first run)...')
        subprocess.check_call(
            ['git', 'clone', 'https://github.com/AniketAslaliya/debateFloor.git', REPO_DIR],
            cwd='/content',
        )
        fresh = True
    os.chdir(REPO_DIR)
    sys.path.insert(0, os.getcwd())
    if fresh:
        subprocess.check_call(PIPS + [
            'trl>=0.12.0', 'transformers>=4.46.0', 'peft', 'accelerate', 'datasets', 'wandb', 'requests', 'matplotlib', 'bitsandbytes',
            'fastapi', 'uvicorn', 'pydantic', 'openenv-core',
        ])
        subprocess.check_call(PIPS + [
            'unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git',
        ])
        return
    try:
        importlib.import_module('app.main')
    except Exception:
        print('Installing env-server deps (openenv-core, fastapi, ...)')
        subprocess.check_call(PIPS + ['openenv-core', 'fastapi', 'uvicorn', 'pydantic'])


ensure_repo_and_deps()
print('CWD:', os.getcwd())

# EDIT: WANDB_API_KEY = wandb_...  |  HF_TOKEN = hf_...  (do not swap)
MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
EPISODES, EPOCHS, BATCH_SIZE = 100, 2, 2
WANDB_API_KEY = ''
WANDB_ENTITY  = 'aniketaslaliya-lnmiit'
HF_TOKEN = ''

os.environ['ENV_BASE_URL'] = 'http://127.0.0.1:7860'
os.environ['WANDB_API_KEY']  = WANDB_API_KEY
os.environ['WANDB_ENTITY']   = WANDB_ENTITY
os.environ['HF_TOKEN']         = HF_TOKEN

importlib.import_module('app.main')
print('app.main: import OK — openenv-core present')

log_path = '/tmp/uvicorn_debatefloor.log'
with open(log_path, 'w') as logf:
    server_proc = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '127.0.0.1', '--port', '7860'],
        cwd=os.getcwd(), stdout=logf, stderr=subprocess.STDOUT,
    )

ok = False
for _ in range(40):
    if server_proc.poll() is not None:
        with open(log_path) as f:
            print('uvicorn died. Log:\n', f.read()[:4000])
        raise RuntimeError('uvicorn exited before /health was ready')
    try:
        r = requests.get('http://127.0.0.1:7860/health', timeout=2)
        if r.status_code == 200 and r.json().get('status') == 'healthy':
            print('Env server healthy (pid=%s)' % server_proc.pid)
            ok = True
            break
    except Exception:
        pass
    time.sleep(1)

if not ok:
    with open(log_path) as f:
        print('Log:\n', f.read()[:4000])
    raise RuntimeError('Env server failed to start (see log above)')

print('Model:', MODEL_NAME, '|', EPISODES, 'ep x', EPOCHS, 'epochs, batch', BATCH_SIZE)
print('WandB:', 'on' if WANDB_API_KEY else 'off')

## Cell 3 — Sanity checks
GPU, Unsloth, env server, reward function — all verified before training.

In [ ]:
import os, sys, torch

# Must be repo root — after restart, Colab CWD is /content, so "server" is not on sys.path.
REPO = '/content/debateFloor'
if not os.path.isdir(os.path.join(REPO, 'server')):
    raise FileNotFoundError(
        f'{REPO} missing or not the debatefloor repo. Run the clone + install cell first.'
    )
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

assert torch.cuda.is_available(), 'No GPU — switch Colab runtime to T4 GPU.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('CWD:', os.getcwd())

try:
    from unsloth import FastLanguageModel
    print('Unsloth: available (MR-3 satisfied)')
except ImportError:
    print('WARNING: Unsloth not found — re-run the install cell (pip unsloth), or training uses plain transformers')

from server.claim_generator import generate_episode_pool
eps = generate_episode_pool(count=3)
print(f'Episode pool: {len(eps)} episodes OK')

import requests
r = requests.post('http://127.0.0.1:7860/reset', json={'task_id': 'clean_claim', 'seed': 42}, timeout=5)
sid = r.json()['session_id']
step_r = requests.post('http://127.0.0.1:7860/step', json={
    'session_id': sid,
    'action': {'action_type': 'approve_claim', 'confidence': 'HIGH', 'parameters': {'reason': 'test'}}
}, timeout=5)
print(f'Env /step test: reward={step_r.json()["reward"]:.4f} done={step_r.json()["done"]}')

print('\nAll checks passed — ready to train.')

## Cell 4 — Train with GRPO
Uses Unsloth 4-bit QLoRA on T4 with live HTTP reward from local env server. ~15 min for 100 episodes.

In [ ]:
import os, sys, torch

REPO = '/content/debateFloor'
if os.path.isdir(REPO):
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)

import train.train_minimal as tm

tm.MODEL_NAME   = MODEL_NAME
tm.EPISODES     = EPISODES
tm.EPOCHS       = EPOCHS
tm.BATCH_SIZE   = BATCH_SIZE
tm.USE_WANDB    = bool(WANDB_API_KEY)
tm.WANDB_KEY    = WANDB_API_KEY
tm.WANDB_ENTITY = WANDB_ENTITY
tm.ENV_BASE_URL = 'http://127.0.0.1:7860'

tm.HAS_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
tm.USE_FP16 = torch.cuda.is_available() and not tm.HAS_BF16
tm.DTYPE    = torch.bfloat16 if tm.HAS_BF16 else torch.float16

print(f'dtype: {tm.DTYPE} | Unsloth: {tm.USE_UNSLOTH}')
print(f'Training: {EPISODES} episodes x {EPOCHS} epochs, reward from localhost:7860/step\n')

tm.main()

## Cell 5 — View results inline

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

# Show reward curve
if Path('docs/reward_curve.svg').exists():
    display(Image('docs/reward_curve.svg'))

# Show component shift
if Path('docs/component_shift.svg').exists():
    display(Image('docs/component_shift.svg'))

# Print key numbers
summary = json.loads(Path('reports/training_summary.json').read_text())
rewards = [r['reward'] for r in summary['log_history'] if 'reward' in r and 'step' in r]
if rewards:
    print(f'\nReward — start: {rewards[0]:.3f} → end: {rewards[-1]:.3f} → peak: {max(rewards):.3f}')

cs = summary.get('component_shift', {})
before = cs.get('before', {})
after  = cs.get('after', {})
if before and after:
    print('\nComponent shift (before → after):')
    for k in before:
        b, a = before[k], after[k]
        arrow = '↑' if a > b else '↓' if a < b else '→'
        print(f'  {k:<28} {b:+.3f}  →  {a:+.3f}  {arrow}')

## Cell 6 — Push checkpoint to HuggingFace Hub
Requires `HF_TOKEN` set in Cell 2. Uses merged 16-bit save (CF-5 safe).

In [ ]:
if HF_TOKEN:
    from huggingface_hub import HfApi, login
    login(token=HF_TOKEN)

    hub_name = f'AniketAsla/debatefloor-grpo-{MODEL_NAME.split("/")[-1].lower()}'
    api = HfApi(token=HF_TOKEN)
    api.upload_folder(
        folder_path='./debatefloor_checkpoint',
        repo_id=hub_name,
        repo_type='model',
        commit_message='GRPO training — Unsloth QLoRA, env-connected HTTP reward',
    )
    print(f'Pushed to https://huggingface.co/{hub_name}')

    # Also push updated artifacts back to the repo
    for f in ['docs/reward_curve.svg', 'docs/component_shift.svg', 'reports/training_summary.json']:
        if os.path.exists(f):
            api.upload_file(
                path_or_fileobj=f,
                path_in_repo=f,
                repo_id='AniketAsla/debateFloor',
                repo_type='space',
                commit_message=f'Update {f} after Colab training',
            )
            print(f'  Synced {f} to Space')
else:
    print('HF_TOKEN not set — skipping. Set it in Cell 2 and re-run.')

## Cell 7 — (Optional) Get WandB run URL to paste into README

In [ ]:
if WANDB_API_KEY:
    import wandb
    api = wandb.Api()
    runs = api.runs(f'{WANDB_ENTITY}/debatefloor-insurance-rl', order='-created_at')
    latest = next(iter(runs), None)
    if latest:
        url = f'https://wandb.ai/{WANDB_ENTITY}/debatefloor-insurance-rl/runs/{latest.id}'
        print(f'\n✅ Your specific WandB run URL (paste this into README):')
        print(f'   {url}')
    else:
        print('No runs found in project yet.')
else:
    print('WandB not enabled. Set WANDB_API_KEY in Cell 2 to get a public run URL.')